In [ ]:
#!pip install torch torchvision rasterio transformers segmentation-models-pytorch numpy tqdm scikit-learn

In [ ]:
import os
import json
import random
import numpy as np
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import AutoModel, AutoConfig
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# ==========================================
# 1. CARICAMENTO CONFIGURAZIONE DA JSON
# ==========================================

# Carica il file config.json
with open('config.json', 'r') as f:
    json_config = json.load(f)

# Mappa il JSON nella struttura CONFIG piatta usata dallo script
CONFIG = {
    # Percorsi
    "DATA_DIR": json_config["paths"]['input_dir'],
    # Nota: IMAGES_DIR e MASKS_DIR sono usati internamente da DATA_DIR nello script originale,
    # ma se vuoi essere specifico puoi usarli nella classe Dataset.
    # Per ora lo script originale si aspetta che dentro DATA_DIR ci siano "images" e "masks".
    
    # Parametri Modello e Dati
    "MODEL_NAME": json_config["model_config"]["model_name"],
    "NUM_CLASSES": json_config["data_specs"]["num_classes"],
    "IN_CHANNELS": json_config["data_specs"]["num_channels"],
    "IMG_SIZE": json_config["data_specs"]["img_size"],
    
    # Iperparametri di Training (Ottimizzati per 4GB VRAM)
    "BATCH_SIZE": json_config["training_params"]["batch_size"],
    "GRAD_ACCUM_STEPS": json_config["training_params"]["gradient_accumulation_steps"],
    "LEARNING_RATE": json_config["training_params"]["learning_rate"],
    "EPOCHS": json_config["training_params"]["num_epochs"],
    "NUM_WORKERS": json_config["training_params"]["num_workers"],
    
    # Hardware
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    
    # Normalizzazione (Statistiche calcolate)
    "MEANS": json_config["data_specs"]["normalization"]["means"],
    "STDS": json_config["data_specs"]["normalization"]["stds"]
}

print(f"Configurazione caricata per {CONFIG['MODEL_NAME']}")
print(f"Device: {CONFIG['DEVICE']} | Batch Reale: {CONFIG['BATCH_SIZE'] * CONFIG['GRAD_ACCUM_STEPS']}")

Configurazione caricata per ibm-nasa-geospatial/Prithvi-EO-2.0-100M-TL
Device: cuda | Batch Reale: 16


In [8]:
# ==========================================
# 2. DATASET CUSTOM (Gestione 10 Bande)
# ==========================================
class SentinelDataset(Dataset):
    def __init__(self, file_list, root_dir, transform=None):
        self.root_dir = root_dir
        self.img_dir = os.path.join(root_dir, "images")
        self.mask_dir = os.path.join(root_dir, "masks")
        self.files = file_list  # Usa la lista passata dall'esterno
        
        # Statistiche (dal config)
        self.means = np.array(CONFIG["MEANS"]).reshape(10, 1, 1)
        self.stds = np.array(CONFIG["STDS"]).reshape(10, 1, 1)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]
        
        # 1. Caricamento Immagine (10 Canali)
        img_path = os.path.join(self.img_dir, img_name)
        with rasterio.open(img_path) as src:
            image = src.read() # (Channels, H, W) -> float32 o uint16
            
        # 2. Caricamento Maschera
        mask_path = os.path.join(self.mask_dir, img_name) # Assicura che i nomi corrispondano
        with rasterio.open(mask_path) as src:
            mask = src.read(1) # (H, W) -> int
            
        # 3. Normalizzazione (Manuale per gestire 10 canali)
        image = image.astype(np.float32)
        image = (image - self.means) / (self.stds + 1e-6)
        
        # 4. Data Augmentation Semplice (Flip)
        if random.random() > 0.5:
            image = np.flip(image, axis=2).copy() # Horizontal flip
            mask = np.flip(mask, axis=1).copy()
            
        # Conversione in Tensor
        return torch.from_numpy(image), torch.from_numpy(mask).long()


In [11]:
# ==========================================
# 3. MODELLO (Prithvi Backbone + Decoder)
# ==========================================
class PrithviSegmentationModel(nn.Module):
    def __init__(self, model_name, num_classes, in_channels=10):
        super().__init__()
        
        # FIX 1: Passiamo num_labels=num_classes esplicitamente per evitare l'errore NoneType
        # FIX 2: Aggiungiamo trust_remote_code=True per supportare architetture custom
        self.config = AutoConfig.from_pretrained(
            model_name, 
            num_labels=num_classes, 
            trust_remote_code=True
        )
        
        self.backbone = AutoModel.from_pretrained(
            model_name, 
            trust_remote_code=True
        )
        
        # --- HACK CRITICO: Adattamento da 6 a 10 Canali ---
        # Verifichiamo se il modello usa "patch_embed" (standard ViT) o altro nome
        if hasattr(self.backbone, "patch_embed"):
            patch_embed_layer = self.backbone.patch_embed.proj
        elif hasattr(self.backbone, "embeddings") and hasattr(self.backbone.embeddings, "patch_embeddings"):
            # Alcune varianti ViT usano questa struttura
            patch_embed_layer = self.backbone.embeddings.patch_embeddings.projection
        else:
            # Fallback generico: cerchiamo il primo strato Conv2d
            # Questo è utile se l'architettura interna cambia nome
            print("Attenzione: Layer patch_embed non trovato con nome standard. Cerco il primo Conv2d...")
            patch_embed_layer = None
            for module in self.backbone.modules():
                if isinstance(module, nn.Conv2d):
                    patch_embed_layer = module
                    break
            if patch_embed_layer is None:
                raise AttributeError("Impossibile trovare il layer di embedding Conv2d nel backbone.")

        # Procedura di inflazione dei pesi
        old_weights = patch_embed_layer.weight
        old_bias = patch_embed_layer.bias
        
        new_in_channels = in_channels
        
        # Creiamo il nuovo layer
        new_patch_embed = nn.Conv2d(
            in_channels=new_in_channels,
            out_channels=patch_embed_layer.out_channels,
            kernel_size=patch_embed_layer.kernel_size,
            stride=patch_embed_layer.stride,
            padding=patch_embed_layer.padding
        )
        
        with torch.no_grad():
            # Copia i pesi originali per i primi 6 canali (o quanti ne ha il modello base)
            orig_channels = old_weights.shape[1]
            new_patch_embed.weight[:, :orig_channels, :, :] = old_weights
            
            # Media dei pesi per i nuovi canali
            mean_weight = torch.mean(old_weights, dim=1, keepdim=True)
            for i in range(orig_channels, new_in_channels):
                new_patch_embed.weight[:, i:i+1, :, :] = mean_weight
            
            if old_bias is not None:
                new_patch_embed.bias = old_bias
            
        # Sostituzione del layer nel backbone
        # Dobbiamo ri-assegnarlo all'attributo corretto
        if hasattr(self.backbone, "patch_embed"):
            self.backbone.patch_embed.proj = new_patch_embed
        elif hasattr(self.backbone, "embeddings") and hasattr(self.backbone.embeddings, "patch_embeddings"):
            self.backbone.embeddings.patch_embeddings.projection = new_patch_embed
        else:
             # Se l'abbiamo trovato con il loop, è più complesso sostituirlo in-place senza sapere il nome del genitore.
             # Per Prithvi-100M solitamente è self.backbone.patch_embed.proj
             # Se il codice sopra ha funzionato per trovarlo, assumiamo la struttura standard per ora.
             self.backbone.patch_embed.proj = new_patch_embed

        # --------------------------------------------------
        
        embed_dim = self.config.hidden_size if hasattr(self.config, "hidden_size") else self.config.embed_dim
        
        self.decoder = nn.Sequential(
            nn.Conv2d(embed_dim, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Conv2d(256, num_classes, kernel_size=1)
        )

    def forward(self, x):
        outputs = self.backbone(pixel_values=x)
        last_hidden_state = outputs.last_hidden_state 
        
        B, N, C = last_hidden_state.shape
        # Nota: Prithvi potrebbe avere token speciali (CLS, o registri) o token temporali.
        # Se N non è un quadrato perfetto, dobbiamo gestire i token extra.
        
        # Calcolo dimensione griglia (assumendo N = H*W o N = 1 + H*W)
        num_patches = N
        H_feat = W_feat = int(np.sqrt(num_patches))
        
        if H_feat * W_feat != num_patches:
            # Caso con CLS token o token temporali extra
            # Prithvi 100M TL ha token temporali. 
            # Spesso i token spaziali sono gli ultimi H*W o bisogna escludere il primo.
            # Tentativo euristico: rimuovere il primo token (spesso CLS) se N è dispari e non quadrato
            if (N - 1) == int(np.sqrt(N-1))**2:
                 last_hidden_state = last_hidden_state[:, 1:, :]
                 N = N - 1
                 H_feat = W_feat = int(np.sqrt(N))
            else:
                 # Fallback: forziamo reshape brutale se non capiamo la struttura
                 pass

        features = last_hidden_state.permute(0, 2, 1).view(B, C, H_feat, W_feat)
        features = F.interpolate(features, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
        logits = self.decoder(features)
        
        return logits

In [6]:
# Pesi calcolati approssimativamente sui tuoi dati
# Ordine: Class 0 (Background?), Class 1, Class 2, ... Class 8
# Se la Classe 0 non esiste o è background, gestiscila. Assumo qui classi 1-8.
# Normalizziamo affinché la classe più frequente (2) abbia peso ~1.0

# Counts: [?, 4670, 7057, 4556, 3596, 4751, 2436, 1678, 4273]
# Weights inversi (più è bassa la count, più alto il peso)
class_weights_list = [
    0.1,  # Class 0 (Se è background/nodata spesso si mette a 0 o basso)
    1.5,  # Class 1
    1.0,  # Class 2 (La più frequente = riferimento)
    1.55, # Class 3
    1.96, # Class 4
    1.48, # Class 5
    2.90, # Class 6
    4.20, # Class 7 (La più rara -> Peso più alto!)
    1.65  # Class 8
]

In [12]:
# ==========================================
# 5. TRAINING LOOP
# ==========================================
def train():
    print(f"--- Inizio Procedura Training ---")
    
    # --- A. PREPARAZIONE FILE E SPLIT STRATIFICATO ---
    img_dir = os.path.join(CONFIG["DATA_DIR"], "images")
    if not os.path.exists(img_dir):
        print(f"ERRORE: La cartella {img_dir} non esiste.")
        return

    # Trova tutti i tif
    all_files = sorted([f for f in os.listdir(img_dir) if f.endswith('.tif')])
    print(f"Totale file trovati: {len(all_files)}")

    # Estrazione Label per stratificazione
    try:
        # Tenta di estrarre la classe dal nome file (es. worker_class7_chip.tif -> 7)
        labels = [int(f.split("_class")[1].split("_")[0]) for f in all_files]
        print("Stratificazione attivata: le classi sono state estratte dai nomi dei file.")
    except Exception as e:
        print(f"Attenzione: Impossibile estrarre classi dai nomi ({e}). Uso Random Split semplice.")
        labels = None

    # Esegue lo split 80% Train, 20% Val
    train_files, val_files = train_test_split(
        all_files, 
        test_size=0.2, 
        random_state=42, 
        stratify=labels # Se labels è None, fa random split, altrimenti stratificato
    )
    
    print(f"Training Set: {len(train_files)} immagini")
    print(f"Validation Set: {len(val_files)} immagini")

    # Creazione dei Dataset
    train_dataset = SentinelDataset(train_files, CONFIG["DATA_DIR"])
    val_dataset = SentinelDataset(val_files, CONFIG["DATA_DIR"])
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True, num_workers=CONFIG["NUM_WORKERS"])
    val_loader = DataLoader(val_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=CONFIG["NUM_WORKERS"])
    
    # --- B. DEFINIZIONE PESI (WEIGHTED LOSS) ---
    # Pesi calcolati sulle frequenze delle tue classi (Classi rare pesano di più)
    # Assumiamo 9 classi (0-8)
    weights_tensor = torch.tensor([0.1, 1.5, 1.0, 1.55, 1.96, 1.48, 2.90, 4.20, 1.65]).float()
    weights_tensor = weights_tensor.to(CONFIG["DEVICE"])
    
    # --- C. SETUP MODELLO E LOSS ---
    model = PrithviSegmentationModel(CONFIG["MODEL_NAME"], CONFIG["NUM_CLASSES"], CONFIG["IN_CHANNELS"])
    model.to(CONFIG["DEVICE"])
    
    # Abilitiamo Gradient Checkpointing per salvare VRAM (FONDAMENTALE PER 4GB)
    model.backbone.gradient_checkpointing_enable()
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["LEARNING_RATE"])
    
    # Passiamo i pesi alla Loss Function
    criterion = FocalDiceLoss(weight=weights_tensor) 
    scaler = GradScaler()
    
    # --- D. LOOP EPOCHE ---
    print("--- Inizio Loop Epoche ---")
    for epoch in range(CONFIG["EPOCHS"]):
        model.train()
        epoch_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['EPOCHS']}")
        
        optimizer.zero_grad()
        
        for step, (images, masks) in enumerate(progress_bar):
            images = images.to(CONFIG["DEVICE"])
            masks = masks.to(CONFIG["DEVICE"])
            
            # Context Mixed Precision (FP16)
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
                loss = loss / CONFIG["GRAD_ACCUM_STEPS"] # Normalizza la loss per l'accumulo
            
            # Backward scalato
            scaler.scale(loss).backward()
            
            # Aggiornamento pesi solo ogni N step (Accumulo Gradienti)
            if (step + 1) % CONFIG["GRAD_ACCUM_STEPS"] == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            # Calcolo loss reale per display (moltiplichiamo di nuovo per visualizzare il valore vero)
            epoch_loss += loss.item() * CONFIG["GRAD_ACCUM_STEPS"]
            progress_bar.set_postfix({"loss": loss.item() * CONFIG["GRAD_ACCUM_STEPS"]})
            
        # Fine Epoca
        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1} Completed. Avg Train Loss: {avg_loss:.4f}")
        
        # Salva checkpoint
        save_path = os.path.join(os.path.dirname(CONFIG["DATA_DIR"]), "models")
        os.makedirs(save_path, exist_ok=True)
        torch.save(model.state_dict(), os.path.join(save_path, f"prithvi_epoch_{epoch+1}.pth"))

if __name__ == "__main__":
    train()

--- Inizio Procedura Training ---
Totale file trovati: 33018
Stratificazione attivata: le classi sono state estratte dai nomi dei file.
Training Set: 26414 immagini
Validation Set: 6604 immagini


TypeError: 'NoneType' object cannot be interpreted as an integer